# [5.6] Embedding Retrieval and Function-Calling Controls - Exercises

This notebook is about specialist models whose quality is easy to overstate. A nice nearest neighbor is not enough; a syntactically plausible tool call is not enough. You will build the small measurement contracts that make retrieval and function-calling results trustworthy before reading the CUDA report for BGE, EmbeddingGemma, and FunctionGemma.

The scope is intentionally narrow. This is not the VLM chapter. It is a text retrieval and function-calling controls lab.

In [ ]:
import json
import re
import sys
from dataclasses import dataclass
from pathlib import Path

import torch as t

chapter = "chapter5_modern_architectures"
section = "part6_multimodal_embedding_function_models"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part6_multimodal_embedding_function_models.tests as tests

MAIN = __name__ == "__main__"


@dataclass(frozen=True)
class EmbeddingRetrievalReport:
    top1_accuracy: float
    mean_reciprocal_rank: float
    mean_positive_similarity: float
    mean_hard_negative_similarity: float
    mean_margin: float


@dataclass(frozen=True)
class CentroidProbe:
    labels: t.Tensor
    centroids: t.Tensor


@dataclass(frozen=True)
class FunctionCallReport:
    accuracy: float
    tool_accuracy: float
    abstention_accuracy: float
    hallucination_rate: float


@dataclass(frozen=True)
class ParsedFunctionCall:
    name: str | None
    arguments: dict[str, str]


_FUNCTION_CALL_RE = re.compile(r"call:([A-Za-z0-9_]+)\{([^}]*)\}")
_FUNCTION_ARG_RE = re.compile(r"([A-Za-z0-9_]+):(?:<escape>(.*?)<escape>|([^,{}]+))")

## 1. Masked Mean Pooling

Embedding models often return one vector per token. A simple sentence embedding is the mean over non-padding tokens. Padding must not affect the result.

<details><summary>Expected output</summary>

```text
All tests in `test_mean_pool_embeddings_ignores_padding_and_matches_reference` passed!
```

</details>

<details><summary>Help - padding is not evidence</summary>

If a large padding sentinel changes the pooled vector, your retrieval system is partly measuring sequence length and batching artifacts. Apply the attention mask before summing and divide by the number of real tokens.

</details>

In [ ]:
def mean_pool_embeddings(token_embeddings: t.Tensor, attention_mask: t.Tensor) -> t.Tensor:
    raise NotImplementedError()


tests.test_mean_pool_embeddings_ignores_padding_and_matches_reference(mean_pool_embeddings)

<details><summary>Solution</summary>

```python
def mean_pool_embeddings(token_embeddings, attention_mask):
    if token_embeddings.ndim != 3:
        raise ValueError("token_embeddings must have shape (batch, seq, dim).")
    if attention_mask.shape != token_embeddings.shape[:2]:
        raise ValueError("attention_mask must have shape (batch, seq).")
    mask = attention_mask.to(device=token_embeddings.device, dtype=token_embeddings.dtype)
    weighted = token_embeddings * mask.unsqueeze(-1)
    denom = mask.sum(dim=-1, keepdim=True).clamp_min(1)
    return weighted.sum(dim=1) / denom
```

</details>

## 2. Paired Retrieval with Hard Negatives

Each query should rank its paired candidate above every distractor. The hard-negative margin is often more informative than top-1 alone.

<details><summary>Expected output</summary>

```text
All tests in `test_retrieval_metrics_rank_pairs_and_hard_negative_margin` passed!
```

</details>

<details><summary>Help - normalize before ranking</summary>

Without normalization, a vector with a larger norm can win even if it points in the wrong semantic direction. Cosine similarity should compare directions, not raw magnitudes.

</details>

In [ ]:
def l2_normalize(x: t.Tensor, *, eps: float = 1e-12) -> t.Tensor:
    raise NotImplementedError()


def cosine_similarity_matrix(
    query_embeddings: t.Tensor,
    candidate_embeddings: t.Tensor,
) -> t.Tensor:
    raise NotImplementedError()


def retrieval_ranks(similarity: t.Tensor, target_indices: t.Tensor) -> t.Tensor:
    raise NotImplementedError()


def embedding_retrieval_report(
    query_embeddings: t.Tensor,
    candidate_embeddings: t.Tensor,
    target_indices: t.Tensor,
) -> EmbeddingRetrievalReport:
    raise NotImplementedError()


tests.test_retrieval_metrics_rank_pairs_and_hard_negative_margin(
    cosine_similarity_matrix,
    retrieval_ranks,
    embedding_retrieval_report,
)

<details><summary>Solution</summary>

```python
def l2_normalize(x, *, eps=1e-12):
    if x.ndim == 0:
        raise ValueError("x must have at least one dimension.")
    return x / x.norm(dim=-1, keepdim=True).clamp_min(eps)


def cosine_similarity_matrix(query_embeddings, candidate_embeddings):
    if query_embeddings.ndim != 2 or candidate_embeddings.ndim != 2:
        raise ValueError("embeddings must have shape (items, dim).")
    if query_embeddings.shape[1] != candidate_embeddings.shape[1]:
        raise ValueError("query and candidate embedding dimensions must match.")
    return l2_normalize(query_embeddings) @ l2_normalize(candidate_embeddings).T


def retrieval_ranks(similarity, target_indices):
    if similarity.ndim != 2:
        raise ValueError("similarity must have shape (queries, candidates).")
    if target_indices.shape != (similarity.shape[0],):
        raise ValueError("target_indices must have shape (queries,).")
    order = similarity.argsort(dim=-1, descending=True)
    matches = order.eq(target_indices[:, None])
    return matches.float().argmax(dim=-1).long() + 1


def embedding_retrieval_report(query_embeddings, candidate_embeddings, target_indices):
    similarity = cosine_similarity_matrix(query_embeddings, candidate_embeddings)
    ranks = retrieval_ranks(similarity, target_indices)
    row = t.arange(similarity.shape[0], device=similarity.device)
    positive = similarity[row, target_indices]
    masked = similarity.clone()
    masked[row, target_indices] = -t.inf
    hard_negative = masked.max(dim=-1).values if similarity.shape[1] > 1 else positive.new_zeros(positive.shape)
    return EmbeddingRetrievalReport(
        top1_accuracy=ranks.eq(1).float().mean().item(),
        mean_reciprocal_rank=(1.0 / ranks.float()).mean().item(),
        mean_positive_similarity=positive.mean().item(),
        mean_hard_negative_similarity=hard_negative.mean().item(),
        mean_margin=(positive - hard_negative).mean().item(),
    )
```

</details>

## 3. Centroid Probe

Nearest-centroid probes are weak on purpose. If this probe cannot recover a controlled concept on held-out points, your geometry probably is not ready for a more ornate classifier.

<details><summary>Expected output</summary>

```text
All tests in `test_centroid_probe_recovers_heldout_clusters` passed!
```

</details>

<details><summary>Help - report held-out accuracy</summary>

Training accuracy can be memorization. Held-out points ask whether the embedding geometry has a stable cluster structure rather than a lucky fit to the examples used to define centroids.

</details>

In [ ]:
def fit_centroid_probe(embeddings: t.Tensor, labels: t.Tensor) -> CentroidProbe:
    raise NotImplementedError()


def predict_centroid_probe(embeddings: t.Tensor, probe: CentroidProbe) -> t.Tensor:
    raise NotImplementedError()


def centroid_probe_accuracy(embeddings: t.Tensor, labels: t.Tensor, probe: CentroidProbe) -> float:
    raise NotImplementedError()


tests.test_centroid_probe_recovers_heldout_clusters(
    fit_centroid_probe,
    predict_centroid_probe,
    centroid_probe_accuracy,
)

<details><summary>Solution</summary>

```python
def fit_centroid_probe(embeddings, labels):
    if embeddings.ndim != 2:
        raise ValueError("embeddings must have shape (items, dim).")
    if labels.shape != (embeddings.shape[0],):
        raise ValueError("labels must have shape (items,).")
    unique_labels = labels.unique(sorted=True)
    centroids = t.stack([embeddings[labels == label].mean(dim=0) for label in unique_labels])
    return CentroidProbe(labels=unique_labels, centroids=l2_normalize(centroids))


def predict_centroid_probe(embeddings, probe):
    similarity = l2_normalize(embeddings) @ probe.centroids.to(embeddings.device).T
    return probe.labels.to(embeddings.device)[similarity.argmax(dim=-1)]


def centroid_probe_accuracy(embeddings, labels, probe):
    predictions = predict_centroid_probe(embeddings, probe)
    return predictions.eq(labels).float().mean().item()
```

</details>

## 4. Function-Call Validity

Tool schemas are behavioral constraints. First mask unavailable tools before selection; then score no-call prompts separately so hallucinated tool calls cannot hide inside an aggregate accuracy.

<details><summary>Expected output</summary>

```text
All tests in `test_mask_disallowed_tools_blocks_invalid_logits` passed!
All tests in `test_function_call_report_separates_tool_and_abstention_errors` passed!
```

</details>

<details><summary>Help - abstention is part of tool use</summary>

A model that always calls a tool can look strong on tool-required prompts and fail ordinary no-tool requests. The hallucination rate is the fraction of no-call examples where the model still selected a tool.

</details>

In [ ]:
def mask_disallowed_tools(logits: t.Tensor, allowed_tools: t.Tensor) -> t.Tensor:
    raise NotImplementedError()


def function_call_report(
    logits: t.Tensor,
    labels: t.Tensor,
    *,
    no_call_id: int,
) -> FunctionCallReport:
    raise NotImplementedError()


tests.test_mask_disallowed_tools_blocks_invalid_logits(mask_disallowed_tools)
tests.test_function_call_report_separates_tool_and_abstention_errors(function_call_report)

<details><summary>Solution</summary>

```python
def mask_disallowed_tools(logits, allowed_tools):
    if logits.ndim != 2:
        raise ValueError("logits must have shape (batch, tools).")
    if allowed_tools.shape == (logits.shape[1],):
        allowed = allowed_tools.to(device=logits.device, dtype=t.bool).expand_as(logits)
    elif allowed_tools.shape == logits.shape:
        allowed = allowed_tools.to(device=logits.device, dtype=t.bool)
    else:
        raise ValueError("allowed_tools must have shape (tools,) or (batch, tools).")
    return logits.masked_fill(~allowed, -t.inf)


def function_call_report(logits, labels, *, no_call_id):
    if logits.ndim != 2:
        raise ValueError("logits must have shape (batch, tools).")
    predictions = logits.argmax(dim=-1)
    tool_mask = labels.ne(no_call_id)
    abstain_mask = labels.eq(no_call_id)
    accuracy = predictions.eq(labels).float().mean().item()
    tool_accuracy = predictions[tool_mask].eq(labels[tool_mask]).float().mean().item() if tool_mask.any() else float("nan")
    abstention_accuracy = predictions[abstain_mask].eq(no_call_id).float().mean().item() if abstain_mask.any() else float("nan")
    hallucination_rate = predictions[abstain_mask].ne(no_call_id).float().mean().item() if abstain_mask.any() else float("nan")
    return FunctionCallReport(accuracy, tool_accuracy, abstention_accuracy, hallucination_rate)
```

</details>

## 5. Parse FunctionGemma Calls

A generated call has to be parsed before it can be scored. The parser should recover the function name and fields even when values contain spaces inside `<escape>` delimiters.

<details><summary>Expected output</summary>

```text
All tests in `test_parse_function_call_text_extracts_name_and_arguments` passed!
```

</details>

<details><summary>Help - JSON is the wrong parser here</summary>

The FunctionGemma text span is structured, but it is not JSON. Use the pinned call grammar, then score function name, required arguments, and exact arguments separately.

</details>

In [ ]:
def parse_function_call_text(text: str) -> ParsedFunctionCall:
    raise NotImplementedError()


tests.test_parse_function_call_text_extracts_name_and_arguments(parse_function_call_text)

<details><summary>Solution</summary>

```python
def parse_function_call_text(text):
    match = _FUNCTION_CALL_RE.search(text)
    if match is None:
        return ParsedFunctionCall(name=None, arguments={})
    arguments = {}
    for arg_match in _FUNCTION_ARG_RE.finditer(match.group(2)):
        escaped_value, bare_value = arg_match.group(2), arg_match.group(3)
        value = escaped_value if escaped_value is not None else bare_value
        arguments[arg_match.group(1)] = value.strip()
    return ParsedFunctionCall(name=match.group(1), arguments=arguments)
```

</details>

## 6. Schema-Token Attribution

Schema-token attribution is a signed compatibility score between hidden states and schema vectors. It is a hypothesis generator, not causal evidence.

<details><summary>Expected output</summary>

```text
All tests in `test_schema_token_attribution_matches_dot_products` passed!
```

</details>

<details><summary>Help - do not softmax causality into existence</summary>

Softmaxing these scores can make them look like probabilities. They are only dot products. Use them to choose follow-up ablations or patching targets, not as final proof of mechanism.

</details>

In [ ]:
def schema_token_attribution(hidden_states: t.Tensor, schema_vectors: t.Tensor) -> t.Tensor:
    raise NotImplementedError()


tests.test_schema_token_attribution_matches_dot_products(schema_token_attribution)

<details><summary>Solution</summary>

```python
def schema_token_attribution(hidden_states, schema_vectors):
    if hidden_states.shape[-1] != schema_vectors.shape[-1]:
        raise ValueError("hidden state and schema vector dimensions must match.")
    return hidden_states @ schema_vectors.T
```

</details>

## Whole-Notebook Contract

After the local exercises pass, the section smoke test checks the same contracts together: padding-safe pooling, retrieval, held-out centroid probes, invalid-tool masking, no-call hallucination diagnostics, and schema-vector scores.

<details><summary>Expected output</summary>

```text
All tests in `test_notebook_contract` passed!
```

</details>

In [ ]:
# Uncomment after completing the exercises above.
# from part6_multimodal_embedding_function_models.solutions import run_smoke_test
# tests.test_notebook_contract(run_smoke_test)

## Signature Result

The final cell reads the committed CUDA report so the notebook displays the same evidence as CI. The important part is not just that the report passes; it is what the failures and controls say.

<details><summary>Interpreting the signature result</summary>

BGE and EmbeddingGemma both retrieve the four controlled pairs and both fail the permuted-pair control. FunctionGemma parses and routes tool names perfectly on the deterministic Mobile Actions slice, but exact and required argument accuracy are `0.875`, with four visible failure indices. Treat those failures as data, not noise to hide.

</details>

In [ ]:
def _load_committed_gpu_report() -> dict:
    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    assert gpu["cuda_available"]
    assert gpu["within_vram_budget"]
    return gpu


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


gpu = run_gpu_test()
{
    "bge_top1": gpu["bge_retrieval_top1_accuracy"],
    "bge_permuted_top1": gpu["bge_permuted_top1_accuracy"],
    "embeddinggemma_top1": gpu["embeddinggemma_retrieval_top1_accuracy"],
    "embeddinggemma_permuted_top1": gpu["embeddinggemma_permuted_top1_accuracy"],
    "functiongemma_parse_accuracy": gpu["functiongemma_parse_accuracy"],
    "functiongemma_function_name_accuracy": gpu["functiongemma_function_name_accuracy"],
    "functiongemma_exact_argument_accuracy": gpu["functiongemma_exact_argument_accuracy"],
    "functiongemma_failure_indices": gpu["functiongemma_failure_indices"],
    "peak_vram_gb": round(gpu["peak_vram_gb"], 3),
}

## Limitations

This section does not claim broad multimodal interpretability, VLM token-flow mechanisms, causal schema attribution, or perfect tool use. No-call hallucination is covered by the toy contract, not by the released FunctionGemma eval. The real checkpoint evidence is scoped to controlled text retrieval, Mobile Actions tool-call parsing/routing/argument scoring, and a benign base-model CUDA forward pass.